In [ ]:
import pandas as pd
import numpy as np
#import statsmodels.api as sm

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from scipy import stats
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Load the dataset
df = pd.read_csv("data/pima.csv")
X = df.drop(columns=["diabetes"])
y = df["diabetes"]

In [ ]:
# Display the dataset
print(X)
print(y)

In [ ]:
#If target is categorical
y = y.map({"neg": 0, "pos": 1})

In [ ]:
# Prepare train and test data, ensuring same proportion of class labels in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
# Build model by Maximum Likelihood Estimation
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
feature_names = X_train.columns

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": model.coef_.flatten()
})

print(coef_df)
print("Intercept:", model.intercept_[0])

In [ ]:
model.classes_

In [ ]:
model.predict_proba(X_test)

In [ ]:
y_pred_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
# Python does not have a function specifically for Hosmer-Lemeshow test.
# Define the function
def hosmer_lemeshow_test(y_true, y_prob, g=10):
    data = pd.DataFrame({
        "y": y_true.values,
        "prob": y_prob
    })
    
    data["decile"] = pd.qcut(data["prob"], g, duplicates="drop")
    
    obs = data.groupby("decile")["y"].sum()
    exp = data.groupby("decile")["prob"].sum()
    n = data.groupby("decile")["y"].count()
    
    hl_stat = np.sum((obs - exp) ** 2 / (exp * (1 - exp / n)))
    df = g - 2
    p_value = 1 - stats.chi2.cdf(hl_stat, df)
    
    return hl_stat, p_value

# Call the function
hl_stat, hl_p = hosmer_lemeshow_test(y_test, y_pred_prob)

print("Hosmer–Lemeshow Chi-square:", hl_stat)
print("Hosmer–Lemeshow p-value:", hl_p)

In [ ]:
# Log-likelihood of fitted model
ll_model = -log_loss(y_test, y_pred_prob, normalize=False)

# Null model (predicts overall mean probability)
p_null = np.mean(y_train)
y_null_prob = np.repeat(p_null, len(y_test))

ll_null = -log_loss(y_test, y_null_prob, normalize=False)

mcfadden_r2 = 1 - (ll_model / ll_null)

print("McFadden R-squared:", mcfadden_r2)

In [ ]:
# Class predictions (0/1)
y_pred = model.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual neg", "Actual pos"],
    columns=["Predicted neg", "Predicted pos"]
)
print(cm_df)

In [ ]:
cm

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

In [ ]:
# Predict probability of sample [pregnant=1, glucose=75, pressure=90, triceps=31, insulin=100, mass=30, pedigree=0.1, age=39]
new_sample = np.array([[1, 75, 90, 31, 100, 30, 0.1, 39]])

# 4. Predict the probability of success
# The predict_proba method returns a 2D array where each row 
# has the probabilities for [class 0 (failure), class 1 (success)]
probabilities = model.predict_proba(new_sample)

In [ ]:
probabilities

In [ ]:
# Extract the probability of success (class 1) i.e. diabetes=pos
probability_of_success = probabilities[0, 1]
probability_of_success